# 감정분류 — EDA·실험 종합 보고
> **작성**: 김한솔 · 2026-07-20 · SKN27기 4팀 「빈틈사이」
> **질문**: 챗봇이 사용자의 감정에 맞는 톤으로 답하려면, 무엇으로 감정을 분류해야 하는가?
> **결론 요약**: KcELECTRA **파인튜닝** + 데이터 추가(**+음성+KOTE**, 90,456건) + **확신도 게이트(0.70)**
> → 채팅체 150문장 F1 **0.776**, 게이트 채택분 정확도 **0.831** (미달분은 문맥 아는 LLM이 재분류)

| 단계 | 산출물 | 상세 문서 |
|---|---|---|
| 데이터 이해 | 본셋·4종 데이터 EDA | `EDA_감정분류_김한솔.ipynb` (§11 4모드 히트맵 포함), `EDA_실험데이터_4종_김한솔.ipynb` |
| 실험 ① | 모델×방식 10조합 비교 | `실험1_모델x방식_4조합_Colab.ipynb` → `exp1_results.json` |
| 실험 ② | 데이터 추가 ablation | `실험2_데이터추가_ablation_Colab.ipynb` → `artifacts_ft/final_metrics.json` |
| 배포 | 서비스 탑재 모델 | `artifacts_ft/` + 확신도 게이트(`calibrate_gate.py`) |

## 1. 데이터 지도 — 5종, 각자의 역할

| 데이터 | 규모 | 출처·성격 | 역할 |
|---|---|---|---|
| **감성대화 말뭉치(본셋)** | 58,234 | AI Hub, 크라우드 직접 작성 구어체 | 기본 학습셋 (계보 100% 검증) |
| **음성 감성 대화** | 추가분 | AI Hub, 전사 텍스트 | 실험②에서 추가 → **채택** |
| **웰니스 상담** | 추가분 | AI Hub, 상담 발화 | 실험②에서 추가 → **탈락** (개선 없음) |
| **KOTE** | 추가분 | 온라인 댓글 44라벨→4감정 매핑 | 실험②에서 추가 → **채택** |
| **chat_eval (자체 제작)** | 150 | 실사용 채팅체 문장 | **시험지** — 학습에 미사용, 도메인 시프트 측정 전용 |

핵심 설계: 학습 데이터(2020, 작성체)와 실사용 입력(2026, 채팅체)의 문체 간극이 있으므로,
**모든 실험을 "작성체 시험지 + 채팅체 시험지" 이중 채점**으로 진행 — 순위가 시험지에 따라 뒤집히는지 항상 확인.

## 2. EDA 핵심 발견 (상세: EDA 노트북 2권)

1. **계보 검증**: 원천 58,268 대화 ↔ 정제본 58,234 — **100.000% 일치**, 탈락 17건은 라벨 상충 단일화
2. **품질**: 결측 0 · 중복 0 · 이상치는 근거 두고 보존 → 전량 유실 없이 활용
3. **6감정 → 4모드 병합**: 응답 전략 단위(위로/편들기/축하/리액션)로 합류 — 슬픔+상처, 분노+불안, 당황→일반
4. **감정 신호 보존**: ㅋㅋ/ㅠㅠ/!/… 출현율이 모드별로 뚜렷이 갈림 → 전처리에서 지우지 않음(`clean_keep_signal`)
5. **문체 정합**: 학습 데이터가 사실상 전부 반말·평서체 — 서비스 입력(친구에게 반말)과 일치
6. **4모드 히트맵 4종** (피드백 반영): 매핑 구조·신호 토큰·연령/상황 교차·4×4 혼동 행렬 — `EDA_감정분류_김한솔.ipynb` §11

## 3. 실험 ① — 모델 × 방식 (10조합, 이중 채점)

- 축 1 **모델**: KcELECTRA(구어체 특화) vs KoBERT
- 축 2 **방식**: 얼려쓰기(임베딩+XGBoost, 레시피 4종) vs 파인튜닝
- 판정 기준: 작성체·채팅체 **두 시험지에서 모두** 1위인 조합만 채택

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
for f in fm.fontManager.ttflist:
    if 'Nanum' in f.name or 'CJK' in f.name or 'Malgun' in f.name:
        plt.rcParams['font.family'] = f.name
        break
plt.rcParams['axes.unicode_minus'] = False

exp1 = pd.DataFrame(json.load(open('exp1_results.json', encoding='utf-8'))).T
exp1 = exp1.sort_values('작성체 F1', ascending=False)
display(exp1.round(3))

ax = exp1[['작성체 F1', '채팅체 F1']].plot(kind='barh', figsize=(9, 5), color=['#64B5F6', '#FFB74D'])
ax.invert_yaxis(); ax.set_title('실험① 10조합 — 작성체 vs 채팅체 F1 (둘 다 1위: KcELECTRA 파인튜닝)')
ax.set_xlabel('Macro F1'); plt.tight_layout(); plt.show()
best = exp1.index[0]
print(f'승자: {best} — 작성체 F1 {exp1.loc[best, "작성체 F1"]:.3f} / 채팅체 F1 {exp1.loc[best, "채팅체 F1"]:.3f}')
print('→ 두 시험지 모두 같은 승자: 순위 역전 없음 — 안심 채택.')
print('→ 관찰: 채팅체 성적이 전 조합에서 작성체보다 낮다 = 도메인 시프트 실재 → 실험②의 동기.')

## 4. 실험 ② — 데이터 추가 ablation

승자 조합(KcELECTRA 파인튜닝)을 고정하고, base(본셋)에 데이터를 하나씩 추가해
**오른 것만 채택**: 음성 채택 · 웰니스 탈락 · KOTE 채택 → 최종 학습셋 **90,456건**.

In [ ]:
fm_ = json.load(open('ft_final_metrics.json', encoding='utf-8'))  # 사본 — artifacts_ft/는 gitignore라 성적표만 공유
print('배포 모델 (artifacts_ft):')
print(f"  조합: base{fm_['combo']}  |  lr {fm_['lr']}  |  학습 {fm_['n_train']:,}건")
print(f"  본셋 test F1: {fm_['test_f1']:.3f}")
print(f"  채팅체 150문장 F1: {fm_['chat150_f1']:.3f}  ← 실사용 문체 시험지")

journey = pd.DataFrame({
    '단계': ['얼려쓰기 최고(실험①)', '파인튜닝(실험①)', '+음성+KOTE(실험②·배포)'],
    '작성체/test F1': [0.660, 0.710, round(fm_['test_f1'], 3)],
    '채팅체 F1': [0.522, 0.548, round(fm_['chat150_f1'], 3)],
})
display(journey)
ax = journey.set_index('단계').plot(kind='bar', figsize=(8, 4), rot=0, color=['#64B5F6', '#FFB74D'])
ax.set_title('성능 여정 — 채팅체 F1이 0.52 → 0.78: 데이터 추가가 도메인 시프트를 메움')
ax.set_ylabel('Macro F1'); plt.tight_layout(); plt.show()
print('핵심 서사: 실험①에서 확인한 채팅체 약점(0.55)이 데이터 추가로 0.776까지 —')
print('본셋 test(0.706)보다 채팅체가 더 높아지는 역전 = 추가 데이터가 실사용 문체를 직접 가르쳤다는 증거.')

## 5. 배포 구성 — 모델 혼자 두지 않는다

| 장치 | 값 | 근거 |
|---|---|---|
| **확신도 게이트** | 0.70 미만 → 문맥 아는 LLM 재분류 | 채팅체 150 스윕: 채택률 82.7%, **채택분 정확도 0.831** (`calibrate_gate.py`) |
| **초단문 게이트** | 10자 미만은 직전 감정 유지 (확신 0.90 이상만 급변 허용) | "응 ㅋㅋ" 오분류 방지 / "짜증나!" 4자는 반영 |
| **복합 감정** | 절 분할("~는데") 후 절별 분류 | 파인튜닝 모델 과확신 실측으로 분포 방식 폐기 |
| **3단 폴백** | 모델 → LLM → normal | 장애 시에도 서비스 지속 |
| **라벨 비노출** | 감정은 톤 결정에만 사용 | 오분류의 체감 비용 최소화 (§EDA 10. 한계) |

## 6. 한계와 모니터링
- 문맥 없는 단문 오분류는 서비스 레이어(최근 10턴 컨텍스트·게이트)가 흡수
- 감정 라벨이 매 턴 저장되므로 **월간 분포 vs EDA 분포 비교(드리프트)** · LLM 폴백 발동률 추적 가능 — 추가 개발 없이 쿼리만으로
- 향후: 실사용 입력 누적 시 채팅체 파인튜닝 재실험